In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:

emp_data = [(1,'manish',50000,'IT','m'),
(2,'vikash',60000,'sales','m'),
(3,'raushan',70000,'marketing','m'),
(4,'mukesh',80000,'IT','m'),
(5,'priti',90000,'sales','f'),
(6,'nikita',45000,'marketing','f'),
(7,'ragini',55000,'marketing','f'),
(8,'rashi',100000,'IT','f'),
(9,'aditya',65000,'IT','m'),
(10,'rahul',50000,'marketing','m'),
(11,'rakhi',50000,'IT','f'),
(12,'akhilesh',90000,'sales','m')]

schema = ['id', 'name', 'salary','dept','gender']

df = spark.createDataFrame(emp_data,schema)

In [0]:
df.show()

In [0]:
window  = Window.partitionBy('dept')

df.withColumn('sum', sum(col('salary')).over(window)).show()

In [0]:
df.createOrReplaceTempView("tmp")
spark.sql(""" select * , sum(salary)over(partition by dept ) as total_sum from tmp  """).show()

In [0]:
window = Window.partitionBy('dept').orderBy('salary')

df.withColumn("row_number", row_number().over(window))\
    .withColumn("rank",rank().over(window))\
        .withColumn("dense_rank",dense_rank().over(window))\
            .show()


In [0]:
window = Window.partitionBy('dept','gender').orderBy('salary')

df.withColumn("row_number", row_number().over(window))\
    .withColumn("rank",rank().over(window))\
        .withColumn("dense_rank",dense_rank().over(window))\
            .show()

In [0]:
window = Window.partitionBy('dept','gender').orderBy(col('salary').desc())

df.withColumn('Rank', row_number().over(window)).filter(col('Rank')<=2).show()



# Lead and Lag

In [0]:

product_data = [
(1,"iphone","01-01-2023",1500000),
(2,"samsung","01-01-2023",1100000),
(3,"oneplus","01-01-2023",1100000),
(1,"iphone","01-02-2023",1300000),
(2,"samsung","01-02-2023",1120000),
(3,"oneplus","01-02-2023",1120000),
(1,"iphone","01-03-2023",1600000),
(2,"samsung","01-03-2023",1080000),
(3,"oneplus","01-03-2023",1160000),
(1,"iphone","01-04-2023",1700000),
(2,"samsung","01-04-2023",1800000),
(3,"oneplus","01-04-2023",1170000),
(1,"iphone","01-05-2023",1200000),
(2,"samsung","01-05-2023",980000),
(3,"oneplus","01-05-2023",1175000),
(1,"iphone","01-06-2023",1100000),
(2,"samsung","01-06-2023",1100000),
(3,"oneplus","01-06-2023",1200000)
]

schema = ['product_id','product_name','sales_date','sales']

df = spark.createDataFrame(product_data , schema)


In [0]:
window = Window.partitionBy('product_name').orderBy('sales_date')

df.withColumn("lead", lead(col("sales"),1).over(window))\
    .withColumn("lag",lag(col('sales'),1).over(window))\
        .show()


In [0]:
window = Window.partitionBy('product_name').orderBy('sales_date')
new_df = df.withColumn("psm_sale", lag(col("sales"),1).over(window))
# new_df.show()
new_df.withColumn('net_%_profit',
                  round(((col('sales')-col('psm_sale'))/col('sales'))*100,1)).show()

In [0]:
df.show()

In [0]:
window = Window.partitionBy('sales_date').orderBy('product_id')

df.withColumn("rank",row_number().over(window)).show()

In [0]:
df.createOrReplaceTempView("tmp")
spark.sql(""" select * , lag(sales,1,0) over(partition by product_name order by product_id) as psm_sale from tmp """).show()

# Range and Row

In [0]:
product_data = [
(2,"samsung","01-01-1995",11000),
(1,"iphone","01-02-2023",1300000),
(2,"samsung","01-02-2023",1120000),
(3,"oneplus","01-02-2023",1120000),
(1,"iphone","01-03-2023",1600000),
(2,"samsung","01-03-2023",1080000),
(3,"oneplus","01-03-2023",1160000),
(1,"iphone","01-01-2006",15000),
(1,"iphone","01-04-2023",1700000),
(2,"samsung","01-04-2023",1800000),
(3,"oneplus","01-04-2023",1170000),
(1,"iphone","01-05-2023",1200000),
(2,"samsung","01-05-2023",980000),
(3,"oneplus","01-05-2023",1175000),
(1,"iphone","01-06-2023",1100000),
(3,"oneplus","01-01-2010",23000),
(2,"samsung","01-06-2023",1100000),
(3,"oneplus","01-06-2023",1200000)
]
my_schema = ['product_id','product_name','sales_date','sales']

my_df = spark.createDataFrame(product_data, my_schema)
my_df2 = spark.createDataFrame(product_data, my_schema)

In [0]:
my_df.show()

In [0]:
window = Window.partitionBy('product_id').orderBy('sales_date')

my_df.withColumn('first', first('sales').over(window))\
    .withColumn('latest', last('sales').over(window)).show()

In [0]:
window2 = Window.partitionBy('product_id').orderBy('sales_date').rowsBetween(Window.unboundedPreceding,Window.unboundedFollowing)

my_df.withColumn('first', first('sales').over(window2))\
    .withColumn('latest', last('sales').over(window2)).show()

# PySpark Window Functions — A Visual Guide

This guide explains window functions by showing what Spark actually does to your rows, step by step. No prior context needed — everything is self-contained.

## The dataset we'll use

| product_id | sales_date | sales |
|---|---|---|
| 101 | Jan | 100 |
| 101 | Feb | 150 |
| 101 | Mar | 200 |
| 102 | Jan | 50 |
| 102 | Feb | 80 |

## The code we're decoding

```python
window2 = Window.partitionBy('product_id') \
                .orderBy('sales_date') \
                .rowsBetween(
                    Window.unboundedPreceding,
                    Window.unboundedFollowing
                )

my_df.withColumn('first', first('sales').over(window2)) \
     .withColumn('latest', last('sales').over(window2)) \
     .show()
```

A window function has **four ingredients**, and you'll see this same skeleton in every section below:

```text
PARTITION BY → which group am I in?
ORDER BY     → in what order?
FRAME        → which rows can I see from where I'm sitting?
FUNCTION     → what do I compute from those visible rows?
```

---

## 1. Visualizing `PARTITION BY`

```python
Window.partitionBy('product_id')
```

`partitionBy` slices the DataFrame into **independent groups**. Every window calculation runs *separately* inside each group — a row in one group has zero visibility into another group.

```text
                    Full DataFrame
    ┌─────────────┬────────────┬───────┐
    │ product_id  │ sales_date │ sales │
    ├─────────────┼────────────┼───────┤
    │ 101         │ Jan        │ 100   │
    │ 101         │ Feb        │ 150   │
    │ 101         │ Mar        │ 200   │
    │ 102         │ Jan        │ 50    │
    │ 102         │ Feb        │ 80    │
    └─────────────┴────────────┴───────┘

              partitionBy('product_id')
                        │
          ┌─────────────┴─────────────┐
          ▼                           ▼
   PRODUCT 101                 PRODUCT 102
   ────────────                ────────────
   Jan → 100                   Jan → 50
   Feb → 150                   Feb → 80
   Mar → 200
   (walled off — cannot see    (walled off — cannot see
    product 102's rows)         product 101's rows)
```

**Key point:** whatever `first()`, `last()`, `sum()`, or any window function computes for product 101, it is mathematically incapable of being influenced by product 102's rows. Think of each partition as its own tiny table that the window function operates on in isolation.

---

## 2. Visualizing `ORDER BY`

```python
.orderBy('sales_date')
```

Once you're inside a partition, `orderBy` fixes the **sequence** of rows. This matters enormously for `first()`/`last()`, because those functions are defined by *position*: "first" means "row #1 in this order", "last" means "the final row in this order."

```text
Suppose the raw data for product 101 arrived scrambled:

   BEFORE orderBy('sales_date')      AFTER orderBy('sales_date')
   (arbitrary order)                  (fixed, meaningful order)

   Mar → 200                          Jan → 100   ← position 1
   Jan → 100          ═══════▶        Feb → 150   ← position 2
   Feb → 150                          Mar → 200   ← position 3
```

If you skipped `orderBy`, Spark would still run, but "first" and "last" would be **undefined / unreliable** — Spark could hand you rows in whatever physical order they happen to sit in that partition, which can even change between runs. `orderBy` is what makes "first" and "last" mean something real (earliest date, latest date) instead of "whatever order happened to come out."

---

## 3. Visualizing the window frame — `rowsBetween(unboundedPreceding, unboundedFollowing)`

This is the part that trips people up, so let's slow down. The **frame** answers: *"From the seat of the current row, how far can I look backward and forward?"*

`rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)` means: *look infinitely backward AND infinitely forward* — i.e., see the **whole partition**, no matter which row is currently "in the driver's seat."

Let's process product 101's partition (Jan, Feb, Mar) one current-row-at-a-time:

```text
Current row = Feb

Jan  → 100   [IN WINDOW]
Feb  → 150   [IN WINDOW] ← current row
Mar  → 200   [IN WINDOW]
```

```text
Current row = Jan

Jan  → 100   [IN WINDOW] ← current row
Feb  → 150   [IN WINDOW]
Mar  → 200   [IN WINDOW]
```

```text
Current row = Mar

Jan  → 100   [IN WINDOW]
Feb  → 150   [IN WINDOW]
Mar  → 200   [IN WINDOW] ← current row
```

Notice: **the bracketed `[IN WINDOW]` set never changes.** Whether Jan, Feb, or Mar is "current," the visible frame is always all three rows. That's the entire point of `unboundedPreceding` → `unboundedFollowing`: it collapses the idea of "current row" into irrelevance for frame membership — every row in the partition sees every other row.

Contrast this (just to build intuition) with a frame like `rowsBetween(-1, 1)`, which would only let each row see its immediate neighbor before and after — that frame *would* change per current row. We'll use that exact frame in Practice Question 3 below.

---

## 4. `first('sales')` — what it actually looks at

```python
first('sales').over(window2)
```

Because the frame is the whole partition, `first()` looks at position 1 in the ordered frame — the earliest row — and copies that value onto **every** row.

```text
PRODUCT 101 (ordered by sales_date, full-partition frame)

Jan → 100   ← FIRST(sales) reads this row
Feb → 150
Mar → 200

first('sales').over(window2) = 100, broadcast to ALL rows below:
```

| product_id | sales_date | sales | first |
|---|---|---|---|
| 101 | Jan | 100 | **100** |
| 101 | Feb | 150 | **100** |
| 101 | Mar | 200 | **100** |

Every row gets `100` because every row's frame is the *same* full partition, and the first element of that frame never changes — it's always Jan's value.

---

## 5. `last('sales')` — what it actually looks at

```python
last('sales').over(window2)
```

Symmetrically, `last()` looks at the final position in the ordered frame — Mar — and copies that value onto every row.

```text
PRODUCT 101 (ordered by sales_date, full-partition frame)

Jan → 100
Feb → 150
Mar → 200  ← LAST(sales) reads this row

last('sales').over(window2) = 200, broadcast to ALL rows below:
```

### Full resulting DataFrame

| product_id | sales_date | sales | first | latest |
|---|---|---|---|---|
| 101 | Jan | 100 | 100 | 200 |
| 101 | Feb | 150 | 100 | 200 |
| 101 | Mar | 200 | 100 | 200 |
| 102 | Jan | 50 | 50 | 80 |
| 102 | Feb | 80 | 50 | 80 |

Notice product 102's rows only ever reference `50` and `80` — proof that the two partitions truly never mix.

---

## 6. The complete mental model

```text
                        DataFrame
                           │
                    partitionBy('product_id')
                           │
              ┌────────────┴────────────┐
              ▼                         ▼
        PRODUCT 101                PRODUCT 102
              │                         │
      orderBy('sales_date')     orderBy('sales_date')
              │                         │
      Jan, Feb, Mar               Jan, Feb
              │                         │
   rowsBetween(unbounded,        rowsBetween(unbounded,
       unbounded)                    unbounded)
              │                         │
     frame = {Jan,Feb,Mar}        frame = {Jan,Feb}
              │                         │
        FIRST = 100                FIRST = 50
        LAST  = 200                LAST  = 80
              │                         │
              ▼                         ▼
     every row in 101 gets      every row in 102 gets
     first=100, latest=200      first=50, latest=80
```

Read this top-to-bottom as a pipeline: **split → order → decide what's visible → compute → stamp the result on every row in the group.**

---

## 7. The SQL equivalent

```sql
SELECT
    product_id,
    sales_date,
    sales,
    FIRST_VALUE(sales) OVER (
        PARTITION BY product_id
        ORDER BY sales_date
        ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
    ) AS first,
    LAST_VALUE(sales) OVER (
        PARTITION BY product_id
        ORDER BY sales_date
        ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
    ) AS latest
FROM my_df;
```

| PySpark | SQL |
|---|---|
| `partitionBy()` | `PARTITION BY` |
| `orderBy()` | `ORDER BY` |
| `rowsBetween()` | `ROWS BETWEEN` |
| `first()` | `FIRST_VALUE()` |
| `last()` | `LAST_VALUE()` |

In plain language: PySpark's `Window` object is just a Python-friendly way to *build* the same `OVER (...)` clause that SQL writes directly. Every `.method()` you chain onto `Window` becomes one clause inside the parentheses of `OVER (...)`.

---

## 8. ⚠️ The `LAST_VALUE()` trap

This is the single most common window-function bug, so pay close attention.

**If you don't explicitly specify the frame**, and you only write `PARTITION BY ... ORDER BY ...`, SQL (and Spark) silently applies a **default frame**:

```text
default frame = RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
```

That default frame *grows* as you move through the partition — it always stops at "wherever I currently am." Watch what that does to `LAST_VALUE`:

```text
DEFAULT FRAME (no explicit ROWS BETWEEN):

Current row = 100 (Jan)
Window → [100]
LAST_VALUE → 100

Current row = 150 (Feb)
Window → [100, 150]
LAST_VALUE → 150

Current row = 200 (Mar)
Window → [100, 150, 200]
LAST_VALUE → 200
```

`LAST_VALUE` isn't returning "the true last row of the partition" — it's returning **"the last row of my own personal, still-growing frame,"** which just happens to always be *me*. That's why people are shocked to see `LAST_VALUE` return the *current row's own value* instead of the partition's actual final value (200 for every row).

Now compare with the **explicit full-frame version** from our original query:

```text
EXPLICIT FRAME — ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING:

Current row = 100 (Jan)
Window → [100, 150, 200]
LAST_VALUE → 200

Current row = 150 (Feb)
Window → [100, 150, 200]
LAST_VALUE → 200

Current row = 200 (Mar)
Window → [100, 150, 200]
LAST_VALUE → 200
```

Side by side:

| Current row | Default frame result | Explicit full-frame result |
|---|---|---|
| Jan (100) | 100 ❌ | 200 ✅ |
| Feb (150) | 150 ❌ | 200 ✅ |
| Mar (200) | 200 ✅ (coincidence) | 200 ✅ |

**One subtlety worth knowing:** the default frame's *lower* bound is `UNBOUNDED PRECEDING`, which never moves — it's always the true start of the partition. That's why `FIRST_VALUE` is usually safe even without an explicit frame (it always sees back to row 1). It's specifically the *upper* bound (`CURRENT ROW` by default) that causes the `LAST_VALUE` trap — that bound keeps sliding forward with whichever row is "current," so it can never reach the actual end of the partition unless you tell it to.

**Rule of thumb:** whenever you use `LAST_VALUE` / `last()` (or any function that needs to see the *whole* group), always specify the frame explicitly:
```python
.rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)
```

---

## 9. Comparing the two window definitions

**Version A — no explicit frame:**
```python
Window.partitionBy('product_id').orderBy('sales_date')
```

**Version B — explicit full-partition frame:**
```python
Window.partitionBy('product_id') \
      .orderBy('sales_date') \
      .rowsBetween(
          Window.unboundedPreceding,
          Window.unboundedFollowing
      )
```

```text
                Version A                          Version B
        (relies on default frame)          (explicit full-partition frame)

Jan  frame → [Jan]                    Jan  frame → [Jan, Feb, Mar]
Feb  frame → [Jan, Feb]               Feb  frame → [Jan, Feb, Mar]
Mar  frame → [Jan, Feb, Mar]          Mar  frame → [Jan, Feb, Mar]

     ↓ growing / row-dependent             ↓ fixed / identical for every row
```

Adding `.rowsBetween(...)` is what converts the frame from "grows as I scan through the partition" into "always the entire group, regardless of position." That single addition is the difference between the buggy `LAST_VALUE` behavior in Section 8 and the correct one.

---

## 10. The reusable mental formula

Whenever you see *any* PySpark window function, decode it in this exact order:

```text
PARTITION BY → Which group am I working inside?
ORDER BY     → In what order are rows inside that group arranged?
FRAME        → From my seat, which rows can I actually see
                 (backward / forward / whole group)?
FUNCTION     → Given only the rows I can see, what am I computing?
```

Expanded checklist — ask yourself these four questions in order, every single time:

1. **Group:** if I ignore everything except `partitionBy(...)`, what buckets does this split the data into?
2. **Order:** within one bucket, what does `orderBy(...)` say row #1, #2, #3... are?
3. **Visibility:** does this window have an explicit frame? If not, remember the default is `UNBOUNDED PRECEDING → CURRENT ROW` — a growing frame, not the whole group.
4. **Computation:** given exactly the rows visible in step 3, what value does the function (`first`, `last`, `sum`, `avg`, `rank`, ...) produce?

If you can answer those four questions for any window query, you can predict its output without running it.

---

## 11. Practice questions

Try to predict the output *before* looking at the answer. Cover the answer section with your hand/scroll slowly if you're disciplined about it.

### Question 1 (easy — the default-frame trap, in a new shape)

**Data:**

| product_id | sales_date | sales |
|---|---|---|
| 201 | Jan | 40 |
| 201 | Feb | 60 |
| 201 | Mar | 20 |
| 201 | Apr | 80 |

**Window definition:**
```python
w = Window.partitionBy('product_id').orderBy('sales_date')

df.withColumn('running_total', sum('sales').over(w)).show()
```

**Predict:** what value does `running_total` show for Jan, Feb, Mar, and Apr? (Hint: no `rowsBetween` was specified — what frame applies by default?)

--- ANSWER ---

Because no explicit frame is given, the default frame `UNBOUNDED PRECEDING → CURRENT ROW` applies. The frame *grows* one row at a time, so `sum()` becomes a running/cumulative total:

```text
Jan frame → [40]                → running_total = 40
Feb frame → [40, 60]            → running_total = 100
Mar frame → [40, 60, 20]        → running_total = 120
Apr frame → [40, 60, 20, 80]    → running_total = 200
```

| sales_date | sales | running_total |
|---|---|---|
| Jan | 40 | 40 |
| Feb | 60 | 100 |
| Mar | 20 | 120 |
| Apr | 80 | 200 |

This is the same default-frame mechanic from Section 8 — except here the "growing frame" behavior is exactly what you *want* for a running total, whereas it was a bug for `LAST_VALUE`. Same default frame, opposite outcome, depending on what function you apply to it.

---

### Question 2 (medium — ranking with ties)

**Data:**

| region | rep | deals_closed |
|---|---|---|
| East | Alice | 5 |
| East | Bob | 8 |
| East | Carol | 8 |
| West | Dave | 3 |
| West | Eve | 6 |

**Window definition:**
```python
w = Window.partitionBy('region').orderBy(col('deals_closed').desc())

df.withColumn('rank', rank().over(w)).show()
```

**Predict:** what `rank` does each rep get? Pay special attention to Bob and Carol — they're tied.

--- ANSWER ---

```text
EAST (ordered by deals_closed, descending)
Bob   → 8   [IN WINDOW]  ← rank 1 (tied)
Carol → 8   [IN WINDOW]  ← rank 1 (tied)
Alice → 5   [IN WINDOW]  ← rank 3  (rank() SKIPS 2, because two reps tied for 1st)

WEST (ordered by deals_closed, descending)
Eve  → 6    [IN WINDOW]  ← rank 1
Dave → 3    [IN WINDOW]  ← rank 2
```

| region | rep | deals_closed | rank |
|---|---|---|---|
| East | Bob | 8 | 1 |
| East | Carol | 8 | 1 |
| East | Alice | 5 | 3 |
| West | Eve | 6 | 1 |
| West | Dave | 3 | 2 |

Key lesson: `rank()` leaves a "gap" after ties (1, 1, 3 — no 2 appears). If you wanted no gaps, you'd use `dense_rank()` instead (which would give 1, 1, 2).

---

### Question 3 (harder — a bounded, sliding frame)

**Data:**

| store_id | day | revenue |
|---|---|---|
| S1 | Mon | 100 |
| S1 | Tue | 200 |
| S1 | Wed | 150 |
| S1 | Thu | 300 |
| S1 | Fri | 250 |

**Window definition:**
```python
w = Window.partitionBy('store_id').orderBy('day').rowsBetween(-1, 1)

df.withColumn('moving_avg', avg('revenue').over(w)).show()
```

Here, `-1` means "one row before the current row" and `1` means "one row after the current row" (this is shorthand for `Window.currentRow - 1` and `Window.currentRow + 1`).

**Predict:** what is `moving_avg` for each day? Think carefully about what happens at Mon (nothing precedes it) and Fri (nothing follows it).

--- ANSWER ---

Unlike Sections 3–9, this frame is *not* the whole partition — it only spans 3 rows: one before, the current one, and one after. Crucially, when a neighbor doesn't exist (start or end of the partition), Spark simply averages whatever *is* available — it does not error or insert a null.

```text
Mon → frame = [Mon, Tue]         (no "Sun" exists, so frame just starts here)
                avg(100, 200) = 150

Tue → frame = [Mon, Tue, Wed]
                avg(100, 200, 150) = 150

Wed → frame = [Tue, Wed, Thu]
                avg(200, 150, 300) = 216.67

Thu → frame = [Wed, Thu, Fri]
                avg(150, 300, 250) = 233.33

Fri → frame = [Thu, Fri]          (no day after Fri, so frame just ends here)
                avg(300, 250) = 275
```

| day | revenue | moving_avg |
|---|---|---|
| Mon | 100 | 150.00 |
| Tue | 200 | 150.00 |
| Wed | 150 | 216.67 |
| Thu | 300 | 233.33 |
| Fri | 250 | 275.00 |

This is the sliding-frame idea previewed back in Section 3: unlike `unboundedPreceding → unboundedFollowing`, this frame's contents genuinely change depending on which row is "current" — and at the two edges of the partition, the frame is simply smaller because there's nothing more to include.

---

## Recap: the four questions, forever

```text
PARTITION BY → Which group am I working inside?
ORDER BY     → In what order?
FRAME        → Which rows can I see?
FUNCTION     → What am I calculating from those rows?
```

Every window function you'll ever encounter in PySpark or SQL — no matter how exotic — decomposes into these same four questions.


In [0]:
my_df.createOrReplaceTempView("my_df")
spark.sql(""" SELECT
  product_id,
  sales_date,
  sales,
  FIRST_VALUE(sales) OVER (
    PARTITION BY product_id
    ORDER BY sales_date
    ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
  ) AS first,
  LAST_VALUE(sales) OVER (
    PARTITION BY product_id
    ORDER BY sales_date
    ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
  ) AS latest
FROM my_df;""").show()

## Question for mail to epmployee serving less than 8 hours of work 

In [0]:
emp_data = [(1,"manish","11-07-2023","10:20"),
        (1,"manish","11-07-2023","11:20"),
        (2,"rajesh","11-07-2023","11:20"),
        (1,"manish","11-07-2023","11:50"),
        (2,"rajesh","11-07-2023","13:20"),
        (1,"manish","11-07-2023","19:20"),
        (2,"rajesh","11-07-2023","17:20"),
        (1,"manish","12-07-2023","10:32"),
        (1,"manish","12-07-2023","12:20"),
        (3,"vikash","12-07-2023","09:12"),
        (1,"manish","12-07-2023","16:23"),
        (3,"vikash","12-07-2023","18:08")]

emp_schema = ["id", "name", "date", "time"]
emp_df = spark.createDataFrame(data=emp_data, schema=emp_schema)

emp_df.show()

In [0]:
from pyspark.sql.functions import to_timestamp, concat_ws

# Convert date and time columns to proper timestamp
emp_df = emp_df.withColumn(
    "timestamp",
    to_timestamp(concat_ws(" ", "date", "time"), "dd-MM-yyyy HH:mm")
)

emp_df.createOrReplaceTempView("emp_df")

# Get first and last timestamps for each employee per day, then filter
spark.sql("""
    WITH t AS (
        SELECT 
            id,
            name,
            date,
            time,
            timestamp,
            FIRST_VALUE(timestamp) OVER (
                PARTITION BY id, date 
                ORDER BY timestamp
                ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
            ) AS login_time,
            LAST_VALUE(timestamp) OVER (
                PARTITION BY id, date 
                ORDER BY timestamp
                ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
            ) AS logout_time
        FROM emp_df
    )
    SELECT * 
    FROM t 
    WHERE (logout_time - login_time) < INTERVAL 8 HOURS
""").show()

## last **3** months sales

In [0]:
my_df2.createOrReplaceTempView("my_df2")

In [0]:
my_df2.show(
    
)

In [0]:
spark.sql("""
          select * , round(avg(sales)over(partition by product_id order by sales_date 
          rows between 2 preceding  and current row),2) as avg from my_df2 
          """).show()